In [ ]:
#%pip install pip --upgrade
#%pip install --upgrade pip setuptools wheel

In [ ]:
#%pip install langchain openai neo4j langchain-openai textdistance pandas spacy\
#    langchain-community seaborn openpyxl chromadb markdown bs4 pypdf neo4j-graphrag\
#        https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz

#!/usr/local/bin/python3.11 -m spacy download en_core_web_md
#!python3 -m spacy download en_core_web_md

In [ ]:
import os
import shutil
import json
import re
import pandas as pd
import tiktoken
import math
import spacy
from typing import Set, Any, Union, Dict, List, Tuple, Hashable
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import chromadb
import textdistance
from tqdm.auto import tqdm
from neo4j.exceptions import Neo4jError, ClientError
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import torch
from huggingface_hub import login

# On MAC:
!export HNSWLIB_NO_NATIVE=1

tqdm.pandas()

In [ ]:
# Set model name
model_name="gpt-4o" # "gpt-4o-mini", IDs of OpenAI's FT models "ft:gpt-4o-2024-08-06:..."

DATA_PATH = "./NLquestions"
CYPHER_PATH = "./CypherQueries"
NEO4J_OUTPUTS_PATH = "./Neo4jOutputs"

os.environ["NEO4J_URI"] = "neo4j+s://helix.biodata.di.unimi.it:7687"
os.environ["NEO4J_USERNAME"] = "Text2Cypher"
os.environ["NEO4J_PASSWORD"] = "Text2Cypher"
os.environ["NEO4J_DATABASE"] = "mirnakgt2c"

graph = Neo4jGraph()
print(graph.schema)

***
# Creating the dataset

In [ ]:
import json
import pandas as pd

def load_level(path, level_name):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data).reset_index(drop=True)
    df["ID"] = f"{level_name}/question_" + (df.index + 1).astype(str)
    df["level"] = level_name
    return df

df_nl   = load_level("FTdataset/nodeLevel.json", "nodeLevel")
df_1hop = load_level("FTdataset/1hop.json",      "1hop")
df_2hop = load_level("FTdataset/2hop.json",      "2hop")
df_3hop = load_level("FTdataset/3hop.json",      "3hop")
df_hl = load_level("FTdataset/hardLevel.json",      "hardLevel")

df_all = pd.concat([df_nl, df_1hop, df_2hop, df_3hop, df_hl], ignore_index=True)
# The following is useful to check the length of the longest cypher query and limit the max number of tokens accordingly (this speed ups evaluation)
print("Longest cypher:", df_all["cypher"].str.len().max())
longest = df_all[df_all["cypher"].str.len() == df_all["cypher"].str.len().max()]['cypher']
print(longest)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
tokens = encoding.encode(longest.iloc[0])
MAX_TOKENS = math.ceil(len(tokens) / 100) * 100
print(len(tokens), MAX_TOKENS)

test_df  = df_all.groupby("level", group_keys=False).sample(frac=0.10, random_state=42)
train_df = df_all.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df[['question', 'cypher']].to_json("FTdataset.json", orient="records", indent=2)
train_df.head(n=3)

In [ ]:
test_df.head(n=3)

***
# Pre-processing for building a vector store for performing RAG
## Step-by-step tutorial for Creating a Vector Store with ChromaDB

In [ ]:
def build_skip_set(test_df):
    skip = set()
    for qid in test_df["ID"]:
        skip.add(f"{qid}.txt")
    return skip

skip_files = build_skip_set(test_df)

def load_txt_documents():
    """
    Load .txt documents from the specified directory.
    Returns: List of Document objects
    """
    all_documents = []

    for folder_name in os.listdir(DATA_PATH):
        folder_path = os.path.join(DATA_PATH, folder_name)
        if os.path.isdir(folder_path):
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                if file_name.endswith(".txt") and folder_name + "/" + file_name not in skip_files:
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            text_content = f.read()
                            all_documents.append(Document(
                                page_content=text_content,
                                metadata={"type": "text", "source": folder_name + "/" + file_name}
                            ))
                    except Exception as e:
                        print(f"Error loading TXT file {file_name}: {e}")

    return all_documents

Check if it works.

In [ ]:
documents = load_txt_documents()

bad = [
    d.metadata["source"]
    for d in documents
    if os.path.basename(d.metadata["source"]) in skip_files
]

assert len(bad) == 0, f"LEAKAGE! Indexed test files: {bad[:5]}"

In [ ]:
documents = load_txt_documents()

documents[1].page_content, documents[1].metadata['source']

***

In [ ]:
from openai import OpenAI
import os

os.environ["OPENAI_API_KEY"] = "...YOUR_OPENAI_API_KEY_HERE..."  
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [ ]:
# Re-create collection if it previously existed
try:
    chroma_client.delete_collection("my_collection")
except:
    pass

collection = chroma_client.create_collection(name="my_collection")
texts = [doc.page_content for doc in documents]
ids = [doc.metadata["source"] for doc in documents]
metas = [doc.metadata for doc in documents]

BATCH = 512

for start in range(0, len(texts), BATCH):
    batch_texts = texts[start:start+BATCH]
    batch_ids   = ids[start:start+BATCH]
    batch_metas = metas[start:start+BATCH]

    resp = client.embeddings.create(
        model="text-embedding-3-large",
        input=batch_texts
    )
    batch_embeddings = [item.embedding for item in resp.data]

    collection.add(
        documents=batch_texts,
        ids=batch_ids,
        metadatas=batch_metas,
        embeddings=batch_embeddings
    )

In [ ]:
# Check if it works
collection = chroma_client.get_collection(name="my_collection")
query = "Find all gene names containing 'MIR'"

query_embedding = client.embeddings.create(
    model="text-embedding-3-large",
    input=[query]
).data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i, doc in enumerate(results["documents"][0]):
    print(f"Match {i+1}:")
    print("→", doc)
    print("Filename:", results["metadatas"][0][i]["source"])
    print("Distance:", results["distances"][0][i])
    print("-" * 40)

***
# Metrics

We will use the following metrics to evaluate the Cypher generation ability of LLMs.
* **Jaro-Winkler**
* **Jaccard**
* **Coverage**
* **Pass**:

In [ ]:
def get_jw_distance(string1: str, string2: str) -> float:
    """
    Calculate the Jaro-Winkler distance between two strings.

    The Jaro-Winkler distance is a measure of similarity between two strings.
    The score is normalized such that 0 equates to no similarity and
    1 is an exact match.
    """
    # Call the jaro_winkler function from the textdistance library.
    return textdistance.jaro_winkler(string1, string2)

In [ ]:
from typing import Any, Dict, Hashable, List, Set, Tuple

from typing import Any, Hashable

from collections.abc import Mapping, Iterable
from typing import Any, Hashable

def _norm_value(v: Any) -> Hashable:
    if isinstance(v, str):
        s = v.strip()
        try:
            return float(s)
        except ValueError:
            return s.lower()

    if isinstance(v, (int, float)):
        return float(v)

    if isinstance(v, Mapping):
        return frozenset(_norm_value(x) for x in v.values())

    if isinstance(v, (list, tuple)):
        return frozenset(_norm_value(x) for x in v)

    if isinstance(v, set):
        return frozenset(_norm_value(x) for x in v)

    return v if isinstance(v, Hashable) else str(v)


def rowsim(setL: Set[Hashable], setR: Set[Hashable]) -> float:
    union = setL | setR
    return 1.0 if not union else len(setL & setR) / len(union)

def make_alignment(dictL: List[Dict], dictR: List[Dict]) -> Tuple[List[Set], List[Set]]:
    swap = len(dictL) > len(dictR)

    setViewsL = [{_norm_value(v) for v in row.values()} for row in dictL]
    setViewsR = [{_norm_value(v) for v in row.values()} for row in dictR]
    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    for i in range(len(setViewsL)):
        max_sim, max_j = -1.0, i
        for j in range(i, len(setViewsR)):
            sim = rowsim(setViewsL[i], setViewsR[j])
            if sim > max_sim:
                max_sim, max_j = sim, j
        setViewsR[i], setViewsR[max_j] = setViewsR[max_j], setViewsR[i]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL
    return setViewsL, setViewsR

def df_sim(dictL: List[Dict], dictR: List[Dict], order_sensitive: bool) -> float:
    if order_sensitive:
        view_L = [[_norm_value(v) for v in row.values()] for row in dictL]
        view_R = [[_norm_value(v) for v in row.values()] for row in dictR]
    else:
        view_L, view_R = make_alignment(dictL, dictR)

    totalL = {(i, x) for i, row in enumerate(view_L) for x in set(row)}
    totalR = {(i, x) for i, row in enumerate(view_R) for x in set(row)}

    union = totalL | totalR
    return 1.0 if not union else len(totalL & totalR) / len(union)

def jaccard_df_sim_pair(pair_L, pair_R) -> float:
    cypher_L, dict_L = pair_L
    cypher_R, dict_R = pair_R
    return df_sim(dict_L, dict_R, "order by" in f"{cypher_L}".lower())

In [ ]:
from typing import Any, Dict, Hashable, List, Set, Tuple
from collections.abc import Mapping

def t2c_norm_value(v: Any) -> Hashable:
    if isinstance(v, str):
        s = v.strip()
        try:
            return float(s)
        except ValueError:
            return s.lower()

    if isinstance(v, (int, float)):
        return float(v)

    return v if isinstance(v, Hashable) else str(v)

def t2c_flat_values(v: Any) -> Set[Hashable]:
    vals: Set[Hashable] = set()

    if isinstance(v, (str, int, float)):
        vals.add(t2c_norm_value(v))
        return vals

    if isinstance(v, Mapping):
        for x in v.values():
            vals |= t2c_flat_values(x)
        return vals

    if isinstance(v, (list, tuple, set)):
        for x in v:
            vals |= t2c_flat_values(x)
        return vals

    vals.add(t2c_norm_value(v))
    return vals


def t2c_row_values(row: Dict) -> Set[Hashable]:
    s: Set[Hashable] = set()
    for v in row.values():
        s |= t2c_flat_values(v)
    return s

def t2c_rowsim(setL: Set[Hashable], setR: Set[Hashable]) -> float:
    union = setL | setR
    return 1.0 if not union else len(setL & setR) / len(union)

def t2c_make_alignment(dictL: List[Dict], dictR: List[Dict]) -> Tuple[List[List[Hashable]], List[List[Hashable]]]:
    swap = len(dictL) > len(dictR)

    setViewsL = [list(t2c_row_values(row)) for row in dictL]
    setViewsR = [list(t2c_row_values(row)) for row in dictR]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    for i in range(len(setViewsL)):
        max_sim, max_j = -1.0, i
        for j in range(i, len(setViewsR)):
            sim = t2c_rowsim(set(setViewsL[i]), set(setViewsR[j]))
            if sim > max_sim:
                max_sim, max_j = sim, j
        setViewsR[i], setViewsR[max_j] = setViewsR[max_j], setViewsR[i]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    return setViewsL, setViewsR

def t2c_row_attr_recall(gt_row: Dict, pred_values: Set[Hashable]) -> float:
    num_attrs = len(gt_row)
    if num_attrs == 0:
        return 1.0

    matched = 0
    for v in gt_row.values():
        gt_vals = t2c_flat_values(v)
        if gt_vals & pred_values:
            matched += 1

    return matched / num_attrs

def t2c_df_sim(dictL: List[Dict], dictR: List[Dict], order_sensitive: bool) -> float:
    if order_sensitive:
        lenL, lenR = len(dictL), len(dictR)
        N = max(lenL, lenR)

        if N == 0:
            return 1.0 

        row_scores: List[float] = []

        for i in range(N):
            if i < lenL:
                gt_row = dictL[i]
                pred_vals = t2c_row_values(dictR[i]) if i < lenR else set()
                row_scores.append(t2c_row_attr_recall(gt_row, pred_vals))
            else:
                pred_vals = t2c_row_values(dictR[i])
                row_scores.append(0.0 if pred_vals else 1.0)

        return sum(row_scores) / N

    view_L, view_R = t2c_make_alignment(dictL, dictR)

    total_attrs = 0
    matched_attrs = 0

    for i, gt_row in enumerate(dictL):
        total_attrs += len(gt_row)
        pred_vals = set(view_R[i]) if i < len(view_R) else set()
        row_recall = t2c_row_attr_recall(gt_row, pred_vals)
        matched_attrs += row_recall * len(gt_row)

    if total_attrs == 0:
        return 1.0

    return matched_attrs / total_attrs

def coverage_df_sim_pair(pair_L, pair_R) -> float:
    cypher_L, dict_L = pair_L
    cypher_R, dict_R = pair_R
    order_sensitive = "order by" in f"{cypher_L}".lower()
    return max(t2c_df_sim(dict_L, dict_R, order_sensitive), df_sim(dict_L, dict_R, order_sensitive))


In [ ]:
# Example usage
dict_a = [{"id": 2, "ReturnedValue": "B"}, {"id": 1, "value": "A"}]
dict_b = [{"id": 1, "value": "A"}, {"id": "2", "value": "B", "extraValue": "C"}, {"id": 1, "value": "A"}]

query_a = "MATCH (n:TableA) RETURN n.id AS id, n.value AS value"
query_b = "MATCH (n:TableB) RETURN n.id AS id, n.value AS value"

similarity_score = jaccard_df_sim_pair((query_a, dict_a), (query_b, dict_b))
print(similarity_score)
similarity_score = coverage_df_sim_pair((query_a, dict_a), (query_b, dict_b))
print(similarity_score)

***
# Vanilla

Here, we use the LLM alone to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

For the non-FT model:

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model_name = "LLaMa3-8b"

login(token="...YOUR_HUGGINGFACE_TOKEN_HERE...")

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
model = base_model
model.config.use_cache = False

hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_TOKENS,
    temperature=0.0,
    top_p=0.95,
    do_sample=False
)
llm = HuggingFacePipeline(pipeline=hf_pipe)

For the FT model:

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model_name = "LLaMa3-8b"

login(token="...YOUR_HUGGINGFACE_TOKEN_HERE...")

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained("../finetuning_LLaMa3-8b/llama3_lora_mirnadiseasegoview")
model = PeftModel.from_pretrained(base_model, "../finetuning_LLaMa3-8b/llama3_lora_mirnadiseasegoview")

hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_TOKENS,
    temperature=0.0,
    top_p=0.95,
    do_sample=False
)
llm = HuggingFacePipeline(pipeline=hf_pipe)

In [ ]:
# Generate Cypher statement based on natural language input

cypher_template = """Write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.

Question: {question}
Cypher query:"""

cypher_prompt = ChatPromptTemplate.from_template(cypher_template)

cypher_chain = cypher_prompt | llm | StrOutputParser()

In [ ]:
response = cypher_chain.invoke(
    {
        "question": "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"
    }
).rsplit("\nCypher query: ", 1)[-1]
print(response)

In [ ]:
import re

def _normalize_generated_cypher(cypher) -> str:
    if cypher is None:
        return ""

    if not isinstance(cypher, str):
        cypher = str(cypher)

    cypher = cypher.strip()
    cypher = re.sub(r"^```(?:cypher)?\s*", "", cypher, flags=re.IGNORECASE)
    cypher = re.sub(r"\s*```$", "", cypher)

    cypher = cypher.replace("\r\n", "\n").replace("\r", "\n").strip()

    cypher = cypher.replace("\x08", r"\b")

    cypher = re.sub(r'(?<!\\)\\b', r'\\\\b', cypher)

    return cypher

def save_checkpoint(df_partial: pd.DataFrame, path: str = "eval_checkpoint.csv"):
    tmp = path + ".tmp"
    df_partial.to_csv(tmp, index=False)
    os.replace(tmp, path)

import torch, gc

import signal
from contextlib import contextmanager

class TimeoutError(Exception):
    pass

@contextmanager
def time_limit(seconds: int):
    def handler(signum, frame):
        raise TimeoutError(f"Timeout after {seconds}s")
    old_handler = signal.signal(signal.SIGALRM, handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

def evaluate_cypher_queries(df, graph, cypher_chain, cypher_prompt, k=1, every=10):
    """
    Evaluate cypher queries and update the DataFrame with new evaluation metrics.

    Parameters:
        df (pd.DataFrame): DataFrame containing at least the columns "question" and "cypher".
        graph: Neo4jGraph object used to run queries.
        cypher_chain: LLM chain for generating Cypher queries.
        cypher_prompt: Prompt for generating Cypher queries.
    Returns:
        pd.DataFrame: The updated DataFrame with new columns:
        "generated_cypher", "true_data", "eval_data", "jaro_winkler", "pass_1", "pass_3", "jaccard".
    """
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    prompt = []
    i = 0

    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            with time_limit(200):
                true_data = graph.query(_normalize_generated_cypher(row["cypher"]))
                
                # Generate 3 cypher queries and fetch data.
                example_generated_cyphers = []
                example_eval_datas = []
                prompts = []

                for _ in range(k):

                    with torch.inference_mode():
                        cypher = _normalize_generated_cypher(cypher_chain.invoke(
                            {"question": row["question"]}
                        ).rsplit("\nCypher query: ", 1)[-1])
                        formatted_prompt = cypher_prompt.format_messages(question=row["question"])
                        prompts.append({
                            "messages": [
                                {"role": msg.type, "content": msg.content}
                                for msg in formatted_prompt
                            ],
                        })

                    example_generated_cyphers.append(cypher)

                    try:
                        example_eval_datas.append(graph.query(cypher))

                    except ClientError as e:
                        if e.code == "Neo.ClientError.Statement.SyntaxError":
                            print("Cypher SyntaxError:", e)
                            example_eval_datas.append([{"id": "Cypher syntax error"}])

                        elif e.code == "Neo.ClientError.Statement.SemanticError":
                            print("Cypher SemanticError:", e)
                            example_eval_datas.append([{"id": "Cypher semantic error"}])

                        else:
                            print("Cypher ClientError:", e.code, e)
                            example_eval_datas.append([{"id": "Cypher client error"}])

                    except Neo4jError as e:
                        print("Neo4j runtime error:", e)
                        example_eval_datas.append([{"id": "Neo4j runtime error"}])

                    except Exception as e:
                        print("Other error:", type(e), e)
                        example_eval_datas.append([{"id": "Other error"}])
                        
                gc.collect()
                torch.cuda.empty_cache()

                # Compute metrics using the first generated cypher/response.
                jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])
                jaccard = jaccard_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                )
                coverage = coverage_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                )
                passs = 1 if jaccard_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                ) == 1 else 0

                # Append computed values.
                generated_cyphers.append(example_generated_cyphers)
                true_datas.append(true_data)
                eval_datas.append(example_eval_datas)
                jaro_winklers.append(jw_value)
                jaccards.append(jaccard)
                coverages.append(coverage)
                passes.append(passs)
                prompt.append(prompts)

                i += 1  

                
                if (i % every) == 0:
                    df_ckpt = pd.DataFrame({
                        "generated_cypher": generated_cyphers,
                        "true_data": true_datas,
                        "eval_data": eval_datas,
                        "jaro_winkler": jaro_winklers,
                        "jaccard": jaccards,
                        "coverage": coverages,
                        "pass": passes,
                        "prompt": prompt,
                    }, index=df.index[:i])  
                    save_checkpoint(df_ckpt, "eval_checkpoint.csv")
        except TimeoutError as e:
            print(f"TimeoutError for index {index}: {e}")
            
            generated_cyphers.append(["Timeout"] * 3)
            true_datas.append([{"id": "Timeout"}])
            eval_datas.append([{"id": "Timeout"}] * 3)
            jaro_winklers.append(0.0)
            jaccards.append(0.0)
            coverages.append(0.0)
            passes.append(0)
            prompt.append({"messages": []})

    # Add evaluated results as new columns to the DataFrame.
    df["generated_cypher"] = generated_cyphers
    df["true_data"] = true_datas
    df["eval_data"] = eval_datas
    df["jaro_winkler"] = jaro_winklers
    df["jaccard"] = jaccards
    df["coverage"] = coverages
    df["pass"] = passes
    df["prompt"] = prompt

    return df

In [ ]:
df = evaluate_cypher_queries(test_df, graph, cypher_chain, cypher_prompt)
df.head(n=3)

In [ ]:
from pathlib import Path

output_dir = Path(model_name)
output_dir.mkdir(parents=True, exist_ok=True)

df.to_excel(
    output_dir / f"evaluating_text2cypher_{model_name}_vanilla.xlsx",
    index=False
)

df.to_pickle(
    output_dir / f"evaluating_text2cypher_{model_name}_vanilla.pkl"
)

df.head(n=3)

***
# Schema

The schema contains node labels, their properties, and the corresponding relationships. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

*** Routines for representing the schema **

In [ ]:
#  Copyright (c) "Neo4j"
#  Neo4j Sweden AB [https://neo4j.com]
#  #
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#  #
#      https://www.apache.org/licenses/LICENSE-2.0
#  #
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
from __future__ import annotations

from typing import Any, Dict, List, Optional, Tuple

import neo4j
from neo4j import Query
from neo4j.exceptions import ClientError, CypherTypeError

BASE_KG_BUILDER_LABEL = "__KGBuilder__"
BASE_ENTITY_LABEL = "__Entity__"
EXCLUDED_LABELS = ["_Bloom_Perspective_", "_Bloom_Scene_"]
EXCLUDED_RELS = ["_Bloom_HAS_SCENE_"]
EXHAUSTIVE_SEARCH_LIMIT = 1700000
LIST_LIMIT = 128
DISTINCT_VALUE_LIMIT = 10

NODE_PROPERTIES_QUERY = (
    "CALL apoc.meta.data({sample: $SAMPLE}) "
    "YIELD label, other, elementType, type, property "
    "WHERE NOT type = 'RELATIONSHIP' AND elementType = 'node' "
    "AND NOT label IN $EXCLUDED_LABELS "
    "WITH label AS nodeLabel, collect({property:property, type:type}) AS properties "
    "RETURN {label: nodeLabel, properties: properties} AS output"
)

REL_PROPERTIES_QUERY = (
    "CALL apoc.meta.data({sample: $SAMPLE}) "
    "YIELD label, other, elementType, type, property "
    "WHERE NOT type = 'RELATIONSHIP' AND elementType = 'relationship' "
    "AND NOT label in $EXCLUDED_RELS "
    "WITH label AS relType, collect({property:property, type:type}) AS properties "
    "RETURN {type: relType, properties: properties} AS output"
)

REL_QUERY = (
    "CALL apoc.meta.data({sample: $SAMPLE}) "
    "YIELD label, other, elementType, type, property "
    "WHERE type = 'RELATIONSHIP' AND elementType = 'node' "
    "UNWIND other AS other_node "
    "WITH * WHERE NOT label IN $EXCLUDED_LABELS "
    "AND NOT other_node IN $EXCLUDED_LABELS "
    "RETURN {start: label, type: property, end: toString(other_node)} AS output"
)

INDEX_QUERY = (
    "CALL apoc.schema.nodes() YIELD label, properties, type, size, valuesSelectivity "
    "WHERE type = 'RANGE' RETURN *, "
    "size * valuesSelectivity as distinctValues"
)

SCHEMA_COUNTS_QUERY = (
    "CALL apoc.meta.graph({sample: $SAMPLE, maxRels: $MAX_RELS}) "
    "YIELD nodes, relationships "
    "RETURN nodes, [rel in relationships | {name: apoc.any.property(rel, 'type'), "
    "count: apoc.any.property(rel, 'count')}] AS relationships"
)



def _clean_string_values(text: str) -> str:
    """Clean string values for schema.

    Cleans the input text by replacing newline and carriage return characters.

    Args:
        text (str): The input text to clean.

    Returns:
        str: The cleaned text.
    """
    return text.replace("\n", " ").replace("\r", " ")


def _value_sanitize(d: Any) -> Any:
    """Sanitize the input dictionary or list.

    Sanitizes the input by removing embedding-like values,
    lists with more than 128 elements, that are mostly irrelevant for
    generating answers in a LLM context. These properties, if left in
    results, can occupy significant context space and detract from
    the LLM's performance by introducing unnecessary noise and cost.

    Args:
        d (Any): The input dictionary or list to sanitize.

    Returns:
        Any: The sanitized dictionary or list.
    """
    if isinstance(d, dict):
        new_dict = {}
        for key, value in d.items():
            if isinstance(value, dict):
                sanitized_value = _value_sanitize(value)
                if (
                    sanitized_value is not None
                ):  # Check if the sanitized value is not None
                    new_dict[key] = sanitized_value
            elif isinstance(value, list):
                if len(value) < LIST_LIMIT:
                    sanitized_value = _value_sanitize(value)
                    if (
                        sanitized_value is not None
                    ):  # Check if the sanitized value is not None
                        new_dict[key] = sanitized_value
                # Do not include the key if the list is oversized
            else:
                new_dict[key] = value
        return new_dict
    elif isinstance(d, list):
        if len(d) < LIST_LIMIT:
            return [
                _value_sanitize(item) for item in d if _value_sanitize(item) is not None
            ]
        else:
            return None
    else:
        return d


def query_database(
    driver: neo4j.Driver,
    query: str,
    params: Dict[str, Any] = {},
    session_params: Dict[str, Any] = {},
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
) -> List[Dict[str, Any]]:
    """
    Queries the database.

    Args:
        driver (neo4j.Driver):  Neo4j Python driver instance.
        query (str): The cypher query.
        params (Optional[dict[str, Any]]): The query parameters. Defaults to None.
        session_params (Optional[dict[str, Any]]): Parameters to pass to the
            session used for executing the query. Defaults to None.
        database (Optional[str]): The name of the database to connect to. Default is 'neo4j'.
        timeout (Optional[float]): The timeout for transactions in seconds.
                Useful for terminating long-running queries.
                By default, there is no timeout set.
        sanitize (bool): A flag to indicate whether to remove lists with
                more than 128 elements from results. Useful for removing
                embedding-like properties from database responses. Default is False.

    Returns:
        list[dict[str, Any]]: the result of the query in json format.
    """
    if not session_params:
        data = driver.execute_query(
            Query(text=query, timeout=timeout),
            database_=database,
            parameters_=params,
        )
        json_data = [r.data() for r in data.records]
        if sanitize:
            json_data = [_value_sanitize(el) for el in json_data]
        return json_data

    session_params.setdefault("database", database)
    with driver.session(**session_params) as session:
        result = session.run(Query(text=query, timeout=timeout), params)
        json_data = [r.data() for r in result]
        if sanitize:
            json_data = [_value_sanitize(el) for el in json_data]
        return json_data


def get_schema(
    driver: neo4j.Driver,
    is_enhanced: bool = False,
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
    sample: int = 1000,
) -> str:
    """
    Returns the schema of the graph as a string with following format:

    .. code-block:: text

        Node properties:
        Person {id: INTEGER, name: STRING}
        Relationship properties:
        KNOWS {fromDate: DATE}
        The relationships:
        (:Person)-[:KNOWS]->(:Person)

    Args:
        driver (neo4j.Driver): Neo4j Python driver instance.
        is_enhanced (bool): Flag indicating whether to format the schema with
            detailed statistics (True) or in a simpler overview format (False).
        database (Optional[str]): The name of the database to connect to. Default is 'neo4j'.
        timeout (Optional[float]): The timeout for transactions in seconds.
                Useful for terminating long-running queries.
                By default, there is no timeout set.
        sanitize (bool): A flag to indicate whether to remove lists with
                more than 128 elements from results. Useful for removing
                embedding-like properties from database responses. Default is False.
        sample (int): Number of nodes to sample for the apoc.meta.data procedure. Setting sample to -1 will remove sampling.
                Defaults to 1000.


    Returns:
        str: the graph schema information in a serialized format.
    """
    structured_schema = get_structured_schema(
        driver=driver,
        is_enhanced=is_enhanced,
        database=database,
        timeout=timeout,
        sanitize=sanitize,
        sample=sample,
    )
    return format_schema(structured_schema, is_enhanced)




def get_structured_schema(
    driver: neo4j.Driver,
    is_enhanced: bool = False,
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
    sample: int = 1000,
) -> dict[str, Any]:
    """
    Returns the structured schema of the graph.

    Returns a dict with following format:

    .. code:: python

        {
            'node_props': {
                'Person': [{'property': 'id', 'type': 'INTEGER'}, {'property': 'name', 'type': 'STRING'}]
            },
            'rel_props': {
                'KNOWS': [{'property': 'fromDate', 'type': 'DATE'}]
            },
            'relationships': [
                {'start': 'Person', 'type': 'KNOWS', 'end': 'Person'}
            ],
            'metadata': {
                'constraint': [
                    {'id': 7, 'name': 'person_id', 'type': 'UNIQUENESS', 'entityType': 'NODE', 'labelsOrTypes': ['Person'], 'properties': ['id'], 'ownedIndex': 'person_id', 'propertyType': None},
                ],
                'index': [
                    {'label': 'Person', 'properties': ['name'], 'size': 2, 'type': 'RANGE', 'valuesSelectivity': 1.0, 'distinctValues': 2.0},
                ]
            }
        }

    Note:
        The internal structure of the returned dict depends on the apoc.meta.data
        and apoc.schema.nodes procedures.

    Warning:
        Some labels are excluded from the output schema:

        - The `__Entity__` and `__KGBuilder__` node labels which are created by the KG Builder pipeline within this package
        - Some labels related to Bloom internals.

    Args:
        driver (neo4j.Driver): Neo4j Python driver instance.
        is_enhanced (bool): Flag indicating whether to format the schema with
            detailed statistics (True) or in a simpler overview format (False).
        database (Optional[str]): The name of the database to connect to. Default is 'neo4j'.
        timeout (Optional[float]): The timeout for transactions in seconds.
            Useful for terminating long-running queries.
            By default, there is no timeout set.
        sanitize (bool): A flag to indicate whether to remove lists with
            more than 128 elements from results. Useful for removing
            embedding-like properties from database responses. Default is False.
        sample (int): Number of nodes to sample for the apoc.meta.data procedure. Setting sample to -1 will remove sampling.
            Defaults to 1000.

    Returns:
        dict[str, Any]: the graph schema information in a structured format.
    """
    node_properties = [
        data["output"]
        for data in query_database(
            driver=driver,
            query=NODE_PROPERTIES_QUERY,
            params={
                "EXCLUDED_LABELS": EXCLUDED_LABELS
                + [BASE_ENTITY_LABEL, BASE_KG_BUILDER_LABEL],
                "SAMPLE": sample,
            },
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )
    ]

    rel_properties = [
        data["output"]
        for data in query_database(
            driver=driver,
            query=REL_PROPERTIES_QUERY,
            params={"EXCLUDED_RELS": EXCLUDED_RELS, "SAMPLE": sample},
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )
    ]

    relationships = [
        data["output"]
        for data in query_database(
            driver=driver,
            query=REL_QUERY,
            params={
                "EXCLUDED_LABELS": EXCLUDED_LABELS
                + [BASE_ENTITY_LABEL, BASE_KG_BUILDER_LABEL],
                "SAMPLE": sample,
            },
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )
    ]

    # Get constraints and indexes
    try:
        constraint = query_database(
            driver=driver,
            query="SHOW CONSTRAINTS",
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )
        index = query_database(
            driver=driver,
            query=INDEX_QUERY,
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )
    except ClientError:
        constraint = []
        index = []

    structured_schema = {
        "node_props": {el["label"]: el["properties"] for el in node_properties},
        "rel_props": {el["type"]: el["properties"] for el in rel_properties},
        "relationships": relationships,
        "metadata": {"constraint": constraint, "index": index},
    }
    if is_enhanced:
        enhance_schema(
            driver=driver,
            structured_schema=structured_schema,
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )
    return structured_schema




def _format_property(prop: Dict[str, Any]) -> Optional[str]:
    """
    Format a single property based on its type and available metadata.

    Depending on the property type, this function provides either an example value,
    a range (for numerical and date types), or a list of available options (for strings).
    If the property is a list that exceeds a defined size limit, it is omitted.

    Args:
        prop (Dict[str, Any]): A dictionary containing details of the property,
            including type, values, min/max, and other metadata.

    Returns:
        Optional[str]: A formatted string representing the property details,
        or None if the property should be skipped (e.g., large lists).
    """
    if prop["type"] == "STRING" and prop.get("values"):
        return f'Example: "{_clean_string_values(prop["values"][0])}"'
        #if prop.get("distinct_count", DISTINCT_VALUE_LIMIT + 1) > DISTINCT_VALUE_LIMIT:
        #    return f'Example: "{_clean_string_values(prop["values"][0])}"'
        #else:
        #    return (
        #        "Available options: "
        #        + f"{[_clean_string_values(el) for el in prop['values']]}"
        #    )
    elif prop["type"] in [
        "INTEGER",
        "FLOAT",
        "DATE",
        "DATE_TIME",
        "LOCAL_DATE_TIME",
    ]:
        if prop.get("min") and prop.get("max"):
            return f"Min: {prop['min']}, Max: {prop['max']}"
        else:
            return f'Example: "{prop["values"][0]}"' if prop.get("values") else ""
    elif prop["type"] == "LIST":
        # preferisci item_examples se presente (prodotto da _build_list_clauses)
        examples = prop.get("item_examples") or prop.get("values")
        if examples and isinstance(examples, list) and examples:
            item_example = examples[0]
            return f'Item Example: "{_clean_string_values(str(item_example))}"'
        # fallback sulle dimensioni
        #if prop.get("min_size") or prop.get("max_size"):
        #    return f"Min Size: {prop.get('min_size','?')}, Max Size: {prop.get('max_size','?')}"
        return ""

def _format_properties(property_dict: Dict[str, Any], is_enhanced: bool) -> List[str]:
    """
    Format a collection of properties for nodes or relationships.

    If `is_enhanced` is True, properties are formatted with additional metadata,
    such as example values or min/max statistics. Otherwise, they are presented in
    a more compact form.

    Args:
        property_dict (Dict[str, Any]): A dictionary mapping labels (for nodes or relationships)
            to lists of property definitions.
        is_enhanced (bool): Flag indicating whether to format properties with additional details.

    Returns:
        List[str]: A list of formatted property descriptions.
    """
    formatted_props = []
    if is_enhanced:
        for label, props in property_dict.items():
            formatted_props.append(f"- **{label}**")
            for prop in props:
                example = _format_property(prop)
                if example is not None:
                    formatted_props.append(
                        f"  - `{prop['property']}`: {prop['type']} {example}"
                    )
    else:
        for label, props in property_dict.items():
            props_str = ", ".join(
                [f"{prop['property']}: {prop['type']}" for prop in props]
            )
            formatted_props.append(f"{label} {{{props_str}}}")
    return formatted_props


def _format_relationships(rels: List[Dict[str, Any]]) -> List[str]:
    """
    Format relationships into a structured string representation.

    Args:
        rels (List[dict]): A list of dictionaries, each containing `start`, `type`, and `end`
            to describe a relationship between two entities.

    Returns:
        List[str]: A list of formatted relationship strings.
    """
    return [f"(:{el['start']})-[:{el['type']}]->(:{el['end']})" for el in rels]


def format_schema(schema: Dict[str, Any], is_enhanced: bool) -> str:
    """
    Format the structured schema into a human-readable string.

    Depending on the `is_enhanced` flag, this function either creates a concise
    listing of node labels and relationship types alongside their properties or
    generates an enhanced, more verbose representation with additional details like
    example or available values and min/max statistics. It also includes a formatted
    list of existing relationships.

    Args:
        schema (Dict[str, Any]): The structured schema dictionary, containing
            properties for nodes and relationships as well as relationship definitions.
        is_enhanced (bool): Flag indicating whether to format the schema with
            detailed statistics (True) or in a simpler overview format (False).

    Returns:
        str: A formatted string representation of the graph schema, including
        node properties, relationship properties, and relationship patterns.
    """
    formatted_node_props = _format_properties(schema["node_props"], is_enhanced)
    formatted_rel_props = _format_properties(schema["rel_props"], is_enhanced)
    formatted_rels = _format_relationships(schema["relationships"])
    return "\n".join(
        [
            "Node properties:",
            "\n".join(formatted_node_props),
            "The relationships:",
            "\n".join(formatted_rels),
            "Relationship properties:",
            "\n".join(formatted_rel_props),
        ]
    )




def _build_str_clauses(
    prop_name: str,
    driver: neo4j.Driver,
    label_or_type: str,
    exhaustive: bool,
    prop_index: Optional[List[Any]] = None,
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
) -> Tuple[List[str], List[str]]:
    """
    Build Cypher clauses for string property statistics.

    Constructs and returns the parts of a Cypher query (`WITH` and `RETURN` clauses)
    required to gather statistical information about a string property. Depending on
    property index metadata and whether the query is exhaustive, this function may
    retrieve a distinct set of values directly from an index or a truncated list of
    distinct values from the actual nodes or relationships.

    Args:
        prop_name (str): The name of the string property.
        driver (neo4j.Driver): Neo4j Python driver instance.
        label_or_type (str): The node label or relationship type to query.
        exhaustive (bool): Whether to perform an exhaustive search or a
            sampled query approach.
        prop_index (Optional[List[Any]]): Optional metadata about the property's
            index. If provided, certain optimizations are applied based on
            distinct value limits and index availability.
        database (Optional[str]): The name of the database to connect to. Default is 'neo4j'.
        timeout (Optional[float]): The timeout for transactions in seconds.
            Useful for terminating long-running queries.
            By default, there is no timeout set.
        sanitize (bool): A flag to indicate whether to remove lists with
            more than 128 elements from results. Useful for removing
            embedding-like properties from database responses. Default is False.

    Returns:
        Tuple[List[str], List[str]]:
            A tuple of two lists. The first list contains the `WITH` clauses, and
            the second list contains the corresponding `RETURN` clauses for the
            string property.
    """
    with_clauses = []
    return_clauses = []
    if (
        not exhaustive
        and prop_index
        and prop_index[0].get("size") > 0
        and prop_index[0].get("distinctValues") <= DISTINCT_VALUE_LIMIT
    ):
        distinct_values = query_database(
            driver=driver,
            query=(
                f"CALL apoc.schema.properties.distinct("
                f"'{label_or_type}', '{prop_name}') YIELD value"
            ),
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )[0]["value"]
        return_clauses.append(
            (f"values: {distinct_values}, distinct_count: {len(distinct_values)}")
        )
    else:
        with_clauses.append(
            (
                f"collect(distinct substring(toString(n.`{prop_name}`)"
                f", 0, 50)) AS `{prop_name}_values`"
            )
        )
        if not exhaustive:
            return_clauses.append(f"values: `{prop_name}_values`")
        else:
            return_clauses.append(
                (
                    f"values: `{prop_name}_values`[..{DISTINCT_VALUE_LIMIT}],"
                    f" distinct_count: size(`{prop_name}_values`)"
                )
            )
    return with_clauses, return_clauses


def _build_list_clauses(prop_name: str) -> Tuple[str, str]:
    """
    Build Cypher clauses for list property size statistics and a sample of list items.

    - calcola min/max della dimensione della lista (come prima)
    - costruisce un sample di esempi sugli elementi interni alla lista usando head()
      (prende il primo elemento della lista per ogni nodo e ne raccoglie i distinti)
    - tronca gli esempi a 50 caratteri per sicurezza di spazio
    """
    # min/max size (come prima)
    with_clause_size = (
        f"min(size(n.`{prop_name}`)) AS `{prop_name}_size_min`, "
        f"max(size(n.`{prop_name}`)) AS `{prop_name}_size_max`"
    )

    # sample di elementi: prende head(...) (primo elemento) e lo raccoglie distintamente
    # si usa substring(toString(...), 0, 50) per evitare valori troppo lunghi
    with_clause_items = (
        f"collect(distinct substring(toString(head(n.`{prop_name}`)), 0, 50)) AS `{prop_name}_item_examples`"
    )

    # unisci i WITH in un'unica clausola (l'ordine non è critico)
    with_clause = with_clause_size + ", " + with_clause_items

    # ritorno: item_examples + dimensioni
    return_clause = (
        f"item_examples: `{prop_name}_item_examples`, "
        f"min_size: `{prop_name}_size_min`, max_size: `{prop_name}_size_max`"
    )
    return with_clause, return_clause



def _build_num_date_clauses(
    prop_name: str, exhaustive: bool, prop_index: Optional[List[Any]] = None
) -> Tuple[List[str], List[str]]:
    """
    Build Cypher clauses for numeric and date/datetime property statistics.

    Constructs and returns the parts of a Cypher query (`WITH` and `RETURN` clauses)
    needed to gather statistical information about numeric or date/datetime
    properties. Depending on whether there is an available index or an exhaustive
    approach is required, this may collect a distinct set of values or compute
    minimum, maximum, and distinct counts.

    Args:
        prop_name (str): The name of the numeric or date/datetime property.
        exhaustive (bool): Whether to perform an exhaustive search or a
            sampled query approach.
        prop_index (Optional[List[Any]]): Optional metadata about the property's
            index. If provided and the search is not exhaustive, it can be used
            to optimize the retrieval of distinct values.

    Returns:
        Tuple[List[str], List[str]]:
            A tuple of two lists. The first list contains the `WITH` clauses, and
            the second list contains the corresponding `RETURN` clauses for the
            numeric or date/datetime property.
    """
    with_clauses = []
    return_clauses = []
    if not prop_index and not exhaustive:
        with_clauses.append(
            f"collect(distinct toString(n.`{prop_name}`)) AS `{prop_name}_values`"
        )
        return_clauses.append(f"values: `{prop_name}_values`")
    else:
        with_clauses.append(f"min(n.`{prop_name}`) AS `{prop_name}_min`")
        with_clauses.append(f"max(n.`{prop_name}`) AS `{prop_name}_max`")
        with_clauses.append(
            f"count(distinct n.`{prop_name}`) AS `{prop_name}_distinct`"
        )
        return_clauses.append(
            (
                f"min: toString(`{prop_name}_min`), "
                f"max: toString(`{prop_name}_max`), "
                f"distinct_count: `{prop_name}_distinct`"
            )
        )
    return with_clauses, return_clauses


def get_enhanced_schema_cypher(
    driver: neo4j.Driver,
    structured_schema: Dict[str, Any],
    label_or_type: str,
    properties: List[Dict[str, Any]],
    exhaustive: bool,
    sample_size: int = 5,
    is_relationship: bool = False,
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
) -> str:
    """
    Build a Cypher query for enhanced schema information.

    Constructs and returns a Cypher query string to gather detailed property
    statistics for either nodes or relationships. Depending on whether the target
    entities are below a certain threshold, it may collect exhaustive information
    or simply sample a few records. This query retrieves data such as minimum and
    maximum values, distinct value counts, and sample values.

    Args:
        driver (neo4j.Driver): Neo4j Python driver instance.
        structured_schema (Dict[str, Any]): The current schema information
            including metadata, indexes, and constraints.
        label_or_type (str): The node label or relationship type to query.
        properties (List[Dict[str, Any]]): A list of property definitions for
            the node label or relationship type.
        exhaustive (bool): Whether to perform an exhaustive search or a
            sampled query approach.
        sample_size (int): The number of nodes or relationships to sample when
            exhaustive is False. Defaults to 5.
        is_relationship (bool, optional): Indicates if the query is for
            a relationship type (True) or a node label (False). Defaults to False.
        database (Optional[str]): The name of the database to connect to. Default is 'neo4j'.
        timeout (Optional[float]): The timeout for transactions in seconds.
            Useful for terminating long-running queries.
            By default, there is no timeout set.
        sanitize (bool): A flag to indicate whether to remove lists with
            more than 128 elements from results. Useful for removing
            embedding-like properties from database responses. Default is False.

    Returns:
        str: A Cypher query string that gathers enhanced property metadata.
    """
    if is_relationship:
        match_clause = f"MATCH ()-[n:`{label_or_type}`]->()"
    else:
        match_clause = f"MATCH (n:`{label_or_type}`)"

    with_clauses = []
    return_clauses = []
    output_dict = {}
    if not exhaustive:
        # Sample random nodes if not exhaustive
        match_clause += f" WITH n LIMIT {sample_size}"
    # Build the with and return clauses
    for prop in properties:
        prop_name = prop["property"]
        prop_type = prop["type"]
        # Check if indexed property, we can still do exhaustive
        prop_index = (
            [
                el
                for el in structured_schema["metadata"]["index"]
                if el["label"] == label_or_type
                and el["properties"] == [prop_name]
                and el["type"] == "RANGE"
            ]
            if not exhaustive
            else None
        )
        if prop_type == "STRING":
            str_w_clauses, str_r_clauses = _build_str_clauses(
                prop_name=prop_name,
                driver=driver,
                label_or_type=label_or_type,
                exhaustive=exhaustive,
                prop_index=prop_index,
                database=database,
                timeout=timeout,
                sanitize=sanitize,
            )
            with_clauses += str_w_clauses
            return_clauses += str_r_clauses
        elif prop_type in [
            "INTEGER",
            "FLOAT",
            "DATE",
            "DATE_TIME",
            "LOCAL_DATE_TIME",
        ]:
            num_date_w_clauses, num_date_r_clauses = _build_num_date_clauses(
                prop_name=prop_name, exhaustive=exhaustive, prop_index=prop_index
            )
            with_clauses += num_date_w_clauses
            return_clauses += num_date_r_clauses
        elif prop_type == "LIST":
            list_w_clause, list_r_clause = _build_list_clauses(prop_name=prop_name)
            with_clauses.append(list_w_clause)
            return_clauses.append(list_r_clause)
        elif prop_type in ["BOOLEAN", "POINT", "DURATION"]:
            continue
        output_dict[prop_name] = "{" + return_clauses.pop() + "}"
    if not output_dict:
        return f"{match_clause}\nRETURN {{}} AS output"
    # Combine with and return clauses
    with_clause = "WITH " + ",\n     ".join(with_clauses) if with_clauses else ""
    return_clause = (
        "RETURN {"
        + ", ".join(f"`{k}`: {v}" for k, v in output_dict.items())
        + "} AS output"
    )
    # Combine all parts of the Cypher query
    cypher_query = "\n".join([match_clause, with_clause, return_clause])
    return cypher_query


def enhance_properties(
    driver: neo4j.Driver,
    structured_schema: Dict[str, Any],
    prop_dict: Dict[str, Any],
    is_relationship: bool,
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
) -> None:
    """
    Enhance the structured schema with detailed statistics for a single node label or relationship type.

    For the specified node label or relationship type, this function queries the database to gather
    property statistics such as minimum and maximum values, distinct value counts, and sample values.
    These statistics are then integrated into the provided structured schema, enriching the schema with
    more in-depth information about each property.

    Args:
        driver (neo4j.Driver): A Neo4j Python driver instance used to run queries against the database.
        structured_schema (Dict[str, Any]): A dictionary representing the current structured schema,
            which will be updated with enhanced property statistics.
        prop_dict (Dict[str, Any]): A dictionary containing the name and count of the node label or
            relationship type to be enhanced.
        is_relationship (bool): Indicates whether the properties to be enhanced belong to a relationship
            (True) or a node (False).
        database (Optional[str]): The name of the database to connect to. Default is 'neo4j'.
        timeout (Optional[float]): The timeout for transactions in seconds.
            Useful for terminating long-running queries.
            By default, there is no timeout set.
        sanitize (bool): A flag to indicate whether to remove lists with
            more than 128 elements from results. Useful for removing
            embedding-like properties from database responses. Default is False.

    Returns:
        None
    """
    name = prop_dict["name"]
    count = prop_dict["count"]
    excluded = EXCLUDED_RELS if is_relationship else EXCLUDED_LABELS
    if name in excluded:
        return
    props = structured_schema["rel_props" if is_relationship else "node_props"].get(
        name
    )
    if not props:  # The node has no properties
        return
    enhanced_cypher = get_enhanced_schema_cypher(
        driver=driver,
        structured_schema=structured_schema,
        label_or_type=name,
        properties=props,
        exhaustive=count < EXHAUSTIVE_SEARCH_LIMIT,
        is_relationship=is_relationship,
        database=database,
        timeout=timeout,
        sanitize=sanitize,
    )
    # Due to schema-flexible nature of neo4j errors can happen
    try:
        # Disable the
        # Neo.ClientNotification.Statement.AggregationSkippedNull
        # notifications raised by the use of collect in the enhanced
        # schema query for nodes
        session_params = (
            {"notifications_disabled_categories": ["UNRECOGNIZED"]}
            if not is_relationship
            else {}
        )
        enhanced_info = query_database(
            driver=driver,
            query=enhanced_cypher,
            session_params=session_params,
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )[0]["output"]
        for prop in props:
            if prop["property"] in enhanced_info:
                prop.update(enhanced_info[prop["property"]])
    except CypherTypeError:
        return


def enhance_schema(
    driver: neo4j.Driver,
    structured_schema: Dict[str, Any],
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
) -> None:
    """
    Enhance the structured schema with detailed property statistics.

    For each node label and relationship type in the structured schema, this
    function queries the database to gather additional property statistics such
    as minimum and maximum values, distinct value counts, and sample values.
    These statistics are then merged into the provided structured schema
    dictionary.

    Args:
        driver (neo4j.Driver): Neo4j Python driver instance.
        structured_schema (Dict[str, Any]): The initial structured schema
            containing node and relationship properties, which will be updated
            with enhanced statistics.
        database (Optional[str]): The name of the database to connect to. Default is 'neo4j'.
        timeout (Optional[float]): The timeout for transactions in seconds.
            Useful for terminating long-running queries.
            By default, there is no timeout set.
        sanitize (bool): A flag to indicate whether to remove lists with
            more than 128 elements from results. Useful for removing
            embedding-like properties from database responses. Default is False.

    Returns:
        None
    """
    schema_counts = query_database(
        driver=driver,
        query=SCHEMA_COUNTS_QUERY,
        params={"SAMPLE": -1, "MAX_RELS": 2000000},
        database=database,
        timeout=timeout,
        sanitize=sanitize,
    )
    # Update node info
    for node in schema_counts[0]["nodes"]:
        enhance_properties(
            driver=driver,
            structured_schema=structured_schema,
            prop_dict=node,
            is_relationship=False,
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )
    # Update rel info
    for rel in schema_counts[0]["relationships"]:
        enhance_properties(
            driver=driver,
            structured_schema=structured_schema,
            prop_dict=rel,
            is_relationship=True,
            database=database,
            timeout=timeout,
            sanitize=sanitize,
        )

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
)

structured_enhanced_schema = get_structured_schema(driver, is_enhanced=True, sample=-1)
enhanced_schema = format_schema(structured_enhanced_schema, is_enhanced=True)
print(enhanced_schema)

As RNA-KG is grounded in a ontology-based schema, we can check if our LLM prompt serialization is correct.

In [ ]:
nodes = pd.read_csv("https://rna-kg.biodata.di.unimi.it/views/mirnadiseasego/nodes.csv")
nodes[':LABEL'] = nodes[':LABEL'].str.split(';')
nodes = nodes.explode(':LABEL', ignore_index=True)

In [ ]:
cols = nodes.columns.difference([":LABEL"])
nodes[cols] = nodes[cols].apply(
    lambda c: c.notna() & (c.astype(str).str.strip() != "")
)
df_agg = (
    nodes
    .groupby(":LABEL", as_index=False)
    .any()
)
bool_cols = nodes.columns.difference([":LABEL"])
df_agg = nodes.groupby(":LABEL", as_index=False)[bool_cols].any()
df_agg = df_agg.rename(columns={"Genomic_coordinates:string[]": "Genomic_coordinates", "ID:string": "ID", "Synonym:string[]": "Synonym", "URI:ID": "URI"})

In [ ]:
def validate_node_schema(df_agg, structured_enhanced_schema):
    
    schema_props = {
        label: {p["property"] for p in props}
        for label, props in structured_enhanced_schema["node_props"].items()
    }

    schema_labels = set(schema_props.keys())

    observed_props = {}

    for _, row in df_agg.iterrows():
        label = row[":LABEL"]
        observed_props[label] = {
            col
            for col in df_agg.columns
            if col != ":LABEL" and bool(row[col])
        }

    observed_labels = set(observed_props.keys())

    errors = {
        "labels_missing_in_data": sorted(schema_labels - observed_labels),
        "labels_extra_in_data": sorted(observed_labels - schema_labels),
        "property_mismatches": {}
    }

    for label in sorted(schema_labels & observed_labels):
        declared = schema_props[label]
        print(declared)
        observed = observed_props[label]
        print(observed)

        if declared != observed:
            errors["property_mismatches"][label] = {
                "missing_properties": sorted(declared - observed),
                "extra_properties": sorted(observed - declared)
            }

    return errors
errors = validate_node_schema(df_agg, structured_enhanced_schema)
if (
    not errors["labels_missing_in_data"]
    and not errors["labels_extra_in_data"]
    and not errors["property_mismatches"]
):
    print("✅ Coherent schema")
else:
    print("❌ Schema NOT coherent")

In [ ]:
nodes = pd.read_csv("https://rna-kg.biodata.di.unimi.it/views/mirnadiseasego/nodes.csv")
edges = pd.read_csv("https://rna-kg.biodata.di.unimi.it/views/mirnadiseasego/edges.csv")
kg = pd.merge(nodes[['URI:ID', ':LABEL']], edges, left_on='URI:ID', right_on=':START_ID').drop(columns=['URI:ID', ':START_ID'])
kg = pd.merge(kg, nodes[['URI:ID', ':LABEL']], left_on=':END_ID', right_on='URI:ID').drop(columns=['URI:ID', ':END_ID'])

In [ ]:
schema_rel = kg[[':LABEL_x', ':TYPE', ':LABEL_y']].drop_duplicates()

In [ ]:
import pandas as pd
from typing import Dict, List, Tuple, Any

def compare_induced_with_structured(schema_rel: pd.DataFrame,
                                    structured_schema: Dict[str, Any],
                                    start_col=":LABEL_x",
                                    type_col=":TYPE",
                                    end_col=":LABEL_y",
                                    save_prefix: str = None
                                   ) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str,Any]]:
    """
    Compare data-induced relationship triplets (schema_rel) with structured schema (structured_schema['relationships']).

    Args:
      - schema_rel: DataFrame with columns start_col, type_col, end_col (can be large)
      - structured_schema: dict containing key 'relationships' -> list of {'start','type','end'}
      - start_col,type_col,end_col: column names in schema_rel (default Neo4j export names)
      - save_prefix: optional prefix to save CSV/JSON outputs

    Returns:
      - induced_triplets: DataFrame with columns ['start','type','end','count'] (observed in data)
      - structured_triplets: DataFrame with columns ['start','type','end'] (declared in structured schema)
      - report: dict with summary and sets/lists for differences
    """

    # 1) Normalize schema_rel column names
    df = schema_rel.copy()
    rename_map = {}
    if start_col in df.columns:
        rename_map[start_col] = "start"
    if type_col in df.columns:
        rename_map[type_col] = "type"
    if end_col in df.columns:
        rename_map[end_col] = "end"
    if rename_map:
        df = df.rename(columns=rename_map)

    # keep only necessary columns and drop nulls
    df = df[['start','type','end']].dropna().astype(str)

    # 2) compute induced triplets with counts
    induced_triplets = df.groupby(['start','type','end']).size().reset_index(name='count')
    induced_triplets = induced_triplets.sort_values(by='count', ascending=False).reset_index(drop=True)

    # 3) get structured triplets from structured_schema
    rels = structured_schema.get('relationships', structured_schema if isinstance(structured_schema, list) else [])
    # rels is expected to be list of dicts with keys start,type,end
    structured_rows = []
    for r in rels:
        # be robust to keys named differently
        s = r.get('start') or r.get('from') or r.get('src') or r.get('subject')
        t = r.get('type') or r.get('rel') or r.get('predicate')
        e = r.get('end') or r.get('to') or r.get('dst') or r.get('object')
        if s is None or t is None or e is None:
            continue
        structured_rows.append({'start': str(s), 'type': str(t), 'end': str(e)})
    structured_triplets = pd.DataFrame(structured_rows).drop_duplicates().reset_index(drop=True)

    # 4) compute set differences
    induced_set = set(induced_triplets.apply(lambda r: (r['start'], r['type'], r['end']), axis=1).tolist())
    print(induced_set)
    structured_set = set(structured_triplets.apply(lambda r: (r['start'], r['type'], r['end']), axis=1).tolist())
    print(structured_set)

    in_data_not_in_structured = sorted(list(induced_set - structured_set))
    in_structured_not_in_data = sorted(list(structured_set - induced_set))
    in_both = sorted(list(induced_set & structured_set))

    # 5) summary
    report = {
        'total_induced_triplets': int(len(induced_triplets)),
        'total_structured_triplets': int(len(structured_triplets)),
        'in_data_not_in_structured_count': len(in_data_not_in_structured),
        'in_structured_not_in_data_count': len(in_structured_not_in_data),
        'in_both_count': len(in_both),
        'in_data_not_in_structured_sample': in_data_not_in_structured[:50],
        'in_structured_not_in_data_sample': in_structured_not_in_data[:50],
    }


    return induced_triplets, structured_triplets, report


induced_triplets, structured_triplets, report = compare_induced_with_structured(schema_rel, structured_enhanced_schema, save_prefix="out/edge_schema_cmp")
print(report)

In [ ]:
excluded = {":TYPE"}
property_cols = [c for c in kg.columns if c not in excluded]
schema_rel_props = (
    kg
    .groupby([":TYPE"])[property_cols]
    .apply(lambda x: x.notna().any())
    .reset_index()
)
schema_rel_props[property_cols] = schema_rel_props[property_cols].astype(bool)
schema_rel_props = schema_rel_props.drop(columns=[":LABEL_x", ":LABEL_y"])
schema_rel_props = schema_rel_props.rename(columns={
    "PubMedID:string[]": "PubMedID",
    "GO_evidence:string[]": "GO_evidence",
    "Context:string[]": "Context",
    "Method:string[]": "Method",
    "FDR:double": "FDR",
    "RNAsister_score:double": "RNAsister_score",
    "Interactor:string[]": "Interactor",
    "p-value:double": "p-value",
    "GeneMANIA_weight:double": "GeneMANIA_weight",
    "Source:string[]": "Source",
    "Mutation:string[]": "Mutation",
    "Weighted_CS_score:double": "Weighted_CS_score",
    "TANRIC_score:double": "TANRIC_score",
    "Rfam_score:double": "Rfam_score"
})

In [ ]:
import pandas as pd
from typing import Dict, Any, Tuple, List

def compare_rel_props(schema_rel_props: pd.DataFrame,
                      structured_enh_schema: Dict[str, Any],
                      type_col: str = ":TYPE",
                      save_prefix: str = None
                     ) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Compare observed relationship properties (schema_rel_props) with structured expected properties (structured_enh_schema['rel_props']).

    Args:
      - schema_rel_props: DataFrame with one row per relationship TYPE and boolean columns for properties.
                          Example columns: [':TYPE','PubMedID','Source',...]
      - structured_enh_schema: dict containing key 'rel_props' -> {rel_type: [ {property:..., ...}, ... ] }
      - type_col: name of the relationship-type column in schema_rel_props (':TYPE' or 'TYPE')
      - save_prefix: optional prefix to save outputs (CSV/JSON)

    Returns:
      - comparison_df: DataFrame with per-type comparison results
      - summary: dict with aggregated stats
    """
    rel_props = structured_enh_schema.get("rel_props", {})
    if not isinstance(rel_props, dict):
        raise ValueError("structured_enh_schema['rel_props'] must be a dict mapping rel_type -> list[prop_specs].")

    df = schema_rel_props.copy()

    # normalize type column
    if type_col not in df.columns:
        # fallback: try common alternatives
        for alt in ["TYPE", ":TYPE", "type"]:
            if alt in df.columns:
                type_col = alt
                break
        else:
            raise KeyError(f"Could not find type column in schema_rel_props. Columns: {list(df.columns)}")

    df = df.rename(columns={type_col: "TYPE"})
    prop_cols = [c for c in df.columns if c != "TYPE"]

    # ensure booleans
    df[prop_cols] = df[prop_cols].astype(bool)

    rows = []
    for _, r in df.iterrows():
        rtype = r["TYPE"]

        observed_props = sorted([c for c in prop_cols if bool(r[c])])

        expected_specs = rel_props.get(rtype, [])
        expected_props = sorted([s.get("property") for s in expected_specs if isinstance(s, dict) and "property" in s])

        missing_in_data = sorted([p for p in expected_props if p not in observed_props])
        extra_in_data = sorted([p for p in observed_props if p not in expected_props])

        if len(expected_props) == 0:
            status = "NO_STRUCTURED_META"
        elif missing_in_data and extra_in_data:
            status = "BOTH_MISSING_AND_EXTRA"
        elif missing_in_data:
            status = "MISSING_EXPECTED"
        elif extra_in_data:
            status = "EXTRA_OBSERVED"
        else:
            status = "OK"

        rows.append({
            "TYPE": rtype,
            "expected_props": expected_props,
            "observed_props": observed_props,
            "missing_in_data": missing_in_data,
            "extra_in_data": extra_in_data,
            "status": status
        })

    comparison_df = pd.DataFrame(rows)

    # summary
    observed_types = set(comparison_df["TYPE"].unique().tolist())
    structured_types = set(rel_props.keys())

    summary = {
        "num_types_in_data": len(observed_types),
        "num_types_in_structured": len(structured_types),
        "types_in_data_not_in_structured": sorted(list(observed_types - structured_types)),
        "types_in_structured_not_in_data": sorted(list(structured_types - observed_types)),
        "status_counts": comparison_df["status"].value_counts().to_dict()
    }

    # optional save
    if save_prefix:
        comparison_df.to_csv(f"{save_prefix}_rel_props_comparison.csv", index=False)
        import json
        with open(f"{save_prefix}_rel_props_summary.json", "w") as f:
            json.dump(summary, f, indent=2)

    return comparison_df, summary

comparison_df, summary = compare_rel_props(schema_rel_props, structured_enhanced_schema, type_col=":TYPE")

print(summary)
comparison_df.head(20)

*** end routines

In [ ]:
simple_schema = format_schema(structured_enhanced_schema, is_enhanced=False)
print(simple_schema)

In [ ]:
enhanced_schema = format_schema(structured_enhanced_schema, is_enhanced=True)
print(enhanced_schema)

In [ ]:
# Generate Cypher statement based on natural language input

cypher_template = """Based on the Neo4j graph schema below,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.
{enhanced_schema}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_template(cypher_template)

cypher_chain = ( RunnablePassthrough.assign(
        enhanced_schema=lambda _: enhanced_schema
    ) | cypher_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
def evaluate_cypher_queries_schema(df, graph, cypher_chain, cypher_prompt, schema, k=1, every=10):
    """
    Evaluate cypher queries and update the DataFrame with new evaluation metrics.

    Parameters:
        df (pd.DataFrame): DataFrame containing at least the columns "question" and "cypher".
        graph: Neo4jGraph object used to run queries.
        cypher_chain: LLM chain for generating Cypher queries.
        cypher_prompt: Prompt for generating Cypher queries.
    Returns:
        pd.DataFrame: The updated DataFrame with new columns:
        "generated_cypher", "true_data", "eval_data", "jaro_winkler", "pass_1", "pass_3", "jaccard".
    """
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    prompt = []
    i = 0

    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            with time_limit(200):
                true_data = graph.query(_normalize_generated_cypher(row["cypher"]))
                
                # Generate 3 cypher queries and fetch data.
                example_generated_cyphers = []
                example_eval_datas = []
                prompts = []

                for _ in range(k):

                    with torch.inference_mode():
                        cypher = _normalize_generated_cypher(cypher_chain.invoke(
                            {"question": row["question"]}
                        ).rsplit("\nCypher query: ", 1)[-1])
                        formatted_prompt = cypher_prompt.format_messages(question=row["question"], enhanced_schema=schema)
                        prompts.append({
                            "messages": [
                                {"role": msg.type, "content": msg.content}
                                for msg in formatted_prompt
                            ],
                        })

                    example_generated_cyphers.append(cypher)

                    try:
                        example_eval_datas.append(graph.query(cypher))

                    except ClientError as e:
                        if e.code == "Neo.ClientError.Statement.SyntaxError":
                            print("Cypher SyntaxError:", e)
                            example_eval_datas.append([{"id": "Cypher syntax error"}])

                        elif e.code == "Neo.ClientError.Statement.SemanticError":
                            print("Cypher SemanticError:", e)
                            example_eval_datas.append([{"id": "Cypher semantic error"}])

                        else:
                            print("Cypher ClientError:", e.code, e)
                            example_eval_datas.append([{"id": "Cypher client error"}])

                    except Neo4jError as e:
                        print("Neo4j runtime error:", e)
                        example_eval_datas.append([{"id": "Neo4j runtime error"}])

                    except Exception as e:
                        print("Other error:", type(e), e)
                        example_eval_datas.append([{"id": "Other error"}])
                
                gc.collect()
                torch.cuda.empty_cache()

                # Compute metrics using the first generated cypher/response.
                jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])
                jaccard = jaccard_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                )
                coverage = coverage_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                )
                passs = 1 if jaccard_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                ) == 1 else 0

                # Append computed values.
                generated_cyphers.append(example_generated_cyphers)
                true_datas.append(true_data)
                eval_datas.append(example_eval_datas)
                jaro_winklers.append(jw_value)
                jaccards.append(jaccard)
                coverages.append(coverage)
                passes.append(passs)
                
                prompt.append(prompts)

                i += 1 

                
                if (i % every) == 0:
                    df_ckpt = pd.DataFrame({
                        "generated_cypher": generated_cyphers,
                        "true_data": true_datas,
                        "eval_data": eval_datas,
                        "jaro_winkler": jaro_winklers,
                        "jaccard": jaccards,
                        "coverage": coverages,
                        "pass": passes,
                        "prompt": prompt,
                        
                    }, index=df.index[:i])  
                    save_checkpoint(df_ckpt, "eval_checkpoint.csv")
        except TimeoutError as e:
            print(f"TimeoutError for index {index}: {e}")
            
            generated_cyphers.append(["Timeout"] * 3)
            true_datas.append([{"id": "Timeout"}])
            eval_datas.append([{"id": "Timeout"}] * 3)
            jaro_winklers.append(0.0)
            jaccards.append(0.0)
            coverages.append(0.0)
            passes.append(0)
            
            prompt.append({"messages": []})

    # Add evaluated results as new columns to the DataFrame.
    df["generated_cypher"] = generated_cyphers
    df["true_data"] = true_datas
    df["eval_data"] = eval_datas
    df["jaro_winkler"] = jaro_winklers
    df["jaccard"] = jaccards
    df["coverage"] = coverages
    df["pass"] = passes
    
    df["prompt"] = prompt

    return df

In [ ]:
df = evaluate_cypher_queries_schema(test_df, graph, cypher_chain, cypher_prompt, enhanced_schema)
df.head(n=3)

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_schema.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_schema.pkl")
df.head(n=3)

***
# Examples via RAG

Examples reteieved from a vector store in a RAG fashion are useful to provide context to the LLM. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from neo4j.exceptions import ClientError, Neo4jError

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NLQUESTIONS_PATH = Path("./NLquestions")    
CYPHER_PATH      = Path("./CypherQueries")   

def cypher_file_from_source(meta_source: str) -> Path:
    """
    meta_source: es '3hop/question_3.txt'
    -> CypherQueries/3hop/question_3.txt
    """
    return CYPHER_PATH / Path(meta_source)


# -----------------------------
# RAG: retrieve examples bundle
# -----------------------------
def retrieve_examples_bundle(question: str, n_results: int = 3):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-large",
        input=[question]
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    docs  = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results.get("distances", [[None] * n_results])[0]

    example_ids = []
    example_distances = []
    prompt_parts = []

    for doc, meta, dist in zip(docs, metas, dists):
        meta_source = meta["source"]  # es: "3hop/question_3.txt"
        example_ids.append(meta_source)
        example_distances.append(dist)

        cypher_file = cypher_file_from_source(meta_source.replace("question", "cypher"))
        if not cypher_file.exists():
            raise FileNotFoundError(f"Cypher file not found for source={meta_source}: {cypher_file}")

        with open(cypher_file, "r", encoding="utf-8") as f:
            cypher_query = f.read().strip()
            
        prompt_parts.append(f"[Query natural language] {doc}\n")
        prompt_parts.append(f"[Query Cypher] {cypher_query}\n\n")

    return {
        "examples_text": "".join(prompt_parts),
        "example_ids": example_ids,
        "example_distances": example_distances,
    }


def enrich_with_examples(inputs: dict) -> dict:
    bundle = retrieve_examples_bundle(inputs["question"], n_results=1)
    return {
        **inputs,
        "examples": bundle["examples_text"],                 
        "example_ids": bundle["example_ids"],                
        "example_distances": bundle["example_distances"],    
    }

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from neo4j.exceptions import ClientError, Neo4jError

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NLQUESTIONS_PATH = Path("./NLquestions")    
CYPHER_PATH      = Path("./CypherQueries") 

def cypher_file_from_source(meta_source: str) -> Path:
    """
    meta_source: es '3hop/question_3.txt'
    -> CypherQueries/3hop/question_3.txt
    """
    return CYPHER_PATH / Path(meta_source)


def retrieve_examples_bundle(question: str, n_results: int = 3):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-large",
        input=[question]
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    docs  = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results.get("distances", [[None] * n_results])[0]

    example_ids = []
    example_distances = []
    prompt_parts = []

    for doc, meta, dist in zip(docs, metas, dists):
        meta_source = meta["source"]  # es: "3hop/question_3.txt"
        example_ids.append(meta_source)
        example_distances.append(dist)

        cypher_file = cypher_file_from_source(meta_source.replace("question", "cypher"))
        if not cypher_file.exists():
            raise FileNotFoundError(f"Cypher file not found for source={meta_source}: {cypher_file}")

        with open(cypher_file, "r", encoding="utf-8") as f:
            cypher_query = f.read().strip()
            
        prompt_parts.append(f"[Query natural language] {doc}\n")
        prompt_parts.append(f"[Query Cypher] {cypher_query}\n\n")

    return {
        "examples_text": "".join(prompt_parts),
        "example_ids": example_ids,
        "example_distances": example_distances,
    }


def enrich_with_examples(inputs: dict) -> dict:
    bundle = retrieve_examples_bundle(inputs["question"], n_results=3)
    return {
        **inputs,
        "examples": bundle["examples_text"],                 
        "example_ids": bundle["example_ids"],                
        "example_distances": bundle["example_distances"],    
    }
# Generate Cypher statement based on natural language input

cypher_template = """Based on the examples below that provide useful
patterns for querying the graph database,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.

Examples:
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_template(cypher_template)

cypher_generate = (
    RunnablePassthrough.assign(
        examples=lambda x: retrieve_examples_bundle(x["question"])
    )
    | cypher_prompt
    | llm
    | StrOutputParser()
)


def _run_graph_query_safe(graph, cypher: str):
    try:
        return graph.query(cypher)
    except ClientError as e:
        if e.code == "Neo.ClientError.Statement.SyntaxError":
            return [{"id": "Cypher syntax error"}]
        if e.code == "Neo.ClientError.Statement.SemanticError":
            return [{"id": "Cypher semantic error"}]
        return [{"id": f"Cypher client error ({e.code})"}]
    except Neo4jError:
        return [{"id": "Neo4j runtime error"}]
    except Exception:
        return [{"id": "Other error"}]


def evaluate_cypher_queries_RAG(df: pd.DataFrame, graph, cypher_generate, cypher_prompt, k: int = 1, every=10) -> pd.DataFrame:
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    
    prompt = []
    i = 0

    retrieved_example_ids = []
    retrieved_example_distances = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            with time_limit(200):
                true_data = _run_graph_query_safe(graph, _normalize_generated_cypher(row["cypher"]))

                example_generated_cyphers = []
                example_eval_datas = []
                example_ids_runs = []
                example_dist_runs = []
                prompts = []

                for _ in range(k):
                    enriched = enrich_with_examples({"question": row["question"]})

                    
                    example_ids_runs.append(enriched["example_ids"])
                    example_dist_runs.append(enriched["example_distances"])

                    with torch.inference_mode():

                        formatted_prompt = cypher_prompt.format_messages(
                            question=enriched["question"],
                            examples=enriched["examples"]
                        )
                        prompts.append({
                            "messages": [
                                {"role": msg.type, "content": msg.content}
                                for msg in formatted_prompt
                            ],
                        })

                        cypher = _normalize_generated_cypher(cypher_chain.invoke(
                            {"question": enriched["question"],
                            "examples": enriched["examples"]}
                        ).rsplit("\nCypher query: ", 1)[-1])

                    example_generated_cyphers.append(cypher)
                    example_eval_datas.append(_run_graph_query_safe(graph, cypher))
                        
                gc.collect()
                torch.cuda.empty_cache()
                
                jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])

                jaccard = jaccard_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                )

                coverage = coverage_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                )
                passs = 1 if coverage == 1 else 0

                # append
                generated_cyphers.append(example_generated_cyphers)
                true_datas.append(true_data)
                eval_datas.append(example_eval_datas)
                jaro_winklers.append(jw_value)
                jaccards.append(jaccard)
                coverages.append(coverage)
                passes.append(passs)
                
                prompt.append(prompts)

                retrieved_example_ids.append(example_ids_runs)
                retrieved_example_distances.append(example_dist_runs)

                if (i % every) == 0:
                    n = len(generated_cyphers)
                    df_ckpt = pd.DataFrame({
                        "generated_cypher": generated_cyphers,
                        "true_data": true_datas,
                        "eval_data": eval_datas,
                        "jaro_winkler": jaro_winklers,
                        "jaccard": jaccards,
                        "coverage": coverages,
                        "pass": passes,
                        
                        "retrieved_example_ids": retrieved_example_ids,
                        "retrieved_example_distances": retrieved_example_distances,
                        "prompt": prompt,
                    }, index=df.index[:n])  
                    save_checkpoint(df_ckpt, "eval_checkpoint.csv")
                i += 1
        except TimeoutError as e:
            generated_cyphers.append(["Timeout"] * k)
            true_datas.append([{"id": "Timeout"}])
            eval_datas.append([{"id": "Timeout"}] * k)
            jaro_winklers.append(0.0)
            jaccards.append(0.0)
            coverages.append(0.0)
            passes.append(0)
            
            retrieved_example_ids.append([["Timeout"] * k])
            retrieved_example_distances.append([["Timeout"] * k])
            prompt.append({"messages": []})
            i += 1

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverage
    out["pass"] = passes
    
    out["prompt"] = prompt

    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances

    return out

inputs = {
    "question": "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"
}

enriched = enrich_with_examples(inputs)

prompt_messages = cypher_prompt.format_messages(
    question=enriched["question"],
    examples=enriched["examples"]
)

print("=== PROMPT ===")
for m in prompt_messages:
    print(f"\n[{m.type.upper()}]\n{m.content}")
df = evaluate_cypher_queries_RAG(test_df, graph, cypher_generate, cypher_prompt)
df.head(3)

df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_RAG.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_RAG.pkl")
df.head(n=3)

In [ ]:
inputs = {
    "question": "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"
}

enriched = enrich_with_examples(inputs)

prompt_messages = cypher_prompt.format_messages(
    question=enriched["question"],
    examples=enriched["examples"]
)

print("=== PROMPT ===")
for m in prompt_messages:
    print(f"\n[{m.type.upper()}]\n{m.content}")

In [ ]:
df = evaluate_cypher_queries_RAG(test_df, graph, cypher_generate, cypher_prompt)
df.head(3)

Let's also save the dataframe so that we avoid running it again and we can inspect it analytically.

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_RAG.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_RAG.pkl")
df.head(n=3)

***
# Examples + Output

Examples retrieved from a vector store in a RAG fashion are useful to provide context to the LLM. Moreover, their outputs might help the LLM in inferring the useful portion of the database schema structure needed to improve the generated query. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from neo4j.exceptions import ClientError, Neo4jError
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# -----------------------------
# PATHS
# -----------------------------
NLQUESTIONS_PATH = Path("./NLquestions")       
CYPHER_PATH      = Path("./CypherQueries")     
NEO4J_OUTPUTS    = Path("./Neo4jOutputs")      

# -----------------------------
# Mapping meta["source"] -> file paths
# meta["source"] esempio: "3hop/question_3.txt"
# -----------------------------
def cypher_file_from_source(meta_source: str) -> Path:
    # "3hop/question_3.txt" -> "CypherQueries/3hop/cypher_3.txt"
    rel = Path(meta_source)
    name = rel.name.replace("question", "cypher")          # question_3.txt -> cypher_3.txt
    return CYPHER_PATH / rel.parent / name

def output_file_from_source(meta_source: str) -> Path:
    # "3hop/question_3.txt" -> "Neo4jOutputs/3hop/output_3.txt"
    rel = Path(meta_source)
    name = rel.name.replace("question", "output")          # question_3.txt -> output_3.txt
    return NEO4J_OUTPUTS / rel.parent / name

# -----------------------------
# RAG retrieval (examples + ids + distances + outputs)
# -----------------------------
def retrieve_examples_bundle_with_output(question: str, n_results: int = 3):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-large",
        input=[question]
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    docs  = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results.get("distances", [[None] * n_results])[0]

    example_ids = []
    example_distances = []
    prompt_parts = []

    for doc, meta, dist in zip(docs, metas, dists):
        meta_source = meta["source"]  # es: "3hop/question_3.txt"
        example_ids.append(meta_source)
        example_distances.append(dist)

        cypher_file = cypher_file_from_source(meta_source)
        out_file    = output_file_from_source(meta_source)

        if not cypher_file.exists():
            raise FileNotFoundError(f"Cypher file not found for source={meta_source}: {cypher_file}")
        if not out_file.exists():
            raise FileNotFoundError(f"Neo4j output file not found for source={meta_source}: {out_file}")

        with open(cypher_file, "r", encoding="utf-8") as f:
            cypher_query = f.read().strip()

        with open(out_file, "r", encoding="utf-8") as f:
            neo4j_output = f.read().strip()

        prompt_parts.append(f"[Query natural language] {doc}\n")
        prompt_parts.append(f"[Query Cypher] {cypher_query}\n")
        prompt_parts.append(f"[Output Neo4j] {neo4j_output}\n\n")

    return {
        "examples_text": "".join(prompt_parts),
        "example_ids": example_ids,
        "example_distances": example_distances,
    }

def enrich_with_examples_with_output(inputs: dict) -> dict:
    bundle = retrieve_examples_bundle_with_output(inputs["question"], n_results=3)
    return {
        **inputs,
        "examples": bundle["examples_text"],
        "example_ids": bundle["example_ids"],
        "example_distances": bundle["example_distances"],
    }

 # Generate Cypher statement based on natural language input

cypher_template = """Based on the examples below that provide useful
patterns for querying the graph database (Neo4j outputs are also reported),
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.

Examples:
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_template(cypher_template)

cypher_generate = (
    RunnablePassthrough.assign(
        examples=lambda x: retrieve_examples_bundle_with_output(x["question"])
    )
    | cypher_prompt
    | llm
    | StrOutputParser()
)


# -----------------------------
# Safe Neo4j query
# -----------------------------
def _run_graph_query_safe(graph, cypher: str):
    try:
        return graph.query(cypher)
    except ClientError as e:
        if e.code == "Neo.ClientError.Statement.SyntaxError":
            return [{"id": "Cypher syntax error"}]
        if e.code == "Neo.ClientError.Statement.SemanticError":
            return [{"id": "Cypher semantic error"}]
        return [{"id": f"Cypher client error ({e.code})"}]
    except Neo4jError:
        return [{"id": "Neo4j runtime error"}]
    except Exception:
        return [{"id": "Other error"}]

def evaluate_cypher_queries_RAG_with_output(df, graph, cypher_generate, cypher_prompt, k=1, every=10):
    generated_cyphers, true_datas, eval_datas = [], [], []
    jaro_winklers, jaccards, coverages = [], [], []
    passes = []
    retrieved_example_ids, retrieved_example_distances, prompt = [], [], []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            with time_limit(200):
                true_data = _run_graph_query_safe(graph, _normalize_generated_cypher(row["cypher"]))

                example_generated_cyphers, example_eval_datas = [], []
                example_ids_runs, example_dist_runs, prompts = [], [], []

                for _ in range(k):
                    enriched = enrich_with_examples_with_output({"question": row["question"]})
                    example_ids_runs.append(enriched["example_ids"])
                    example_dist_runs.append(enriched["example_distances"])

                    with torch.inference_mode():

                        formatted_prompt = cypher_prompt.format_messages(
                            question=enriched["question"],
                            examples=enriched["examples"]
                        )
                        prompts.append({
                            "messages": [
                                {"role": msg.type, "content": msg.content}
                                for msg in formatted_prompt
                            ],
                        })

                        cypher = _normalize_generated_cypher(cypher_chain.invoke(
                            {"question": enriched["question"], "examples": enriched["examples"]}
                        ).rsplit("\nCypher query: ", 1)[-1])

                    example_generated_cyphers.append(cypher)
                    example_eval_datas.append(_run_graph_query_safe(graph, cypher))

                jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])
                jaccard = jaccard_df_sim_pair((row["cypher"], true_data),
                                      (example_generated_cyphers[0], example_eval_datas[0]))
                coverage = coverage_df_sim_pair((row["cypher"], true_data),
                                              (example_generated_cyphers[0], example_eval_datas[0]))

                passs = 1 if coverage == 1 else 0

                generated_cyphers.append(example_generated_cyphers)
                true_datas.append(true_data)
                eval_datas.append(example_eval_datas)
                jaro_winklers.append(jw_value)
                jaccards.append(jaccard)
                coverages.append(coverage)
                passes.append(passs)
                retrieved_example_ids.append(example_ids_runs)
                retrieved_example_distances.append(example_dist_runs)
                prompt.append(prompts)

        except TimeoutError:
            generated_cyphers.append(["Timeout"] * k)
            true_datas.append([{"id": "Timeout"}])
            eval_datas.append([{"id": "Timeout"}] * k)
            jaro_winklers.append(0.0)
            jaccards.append(0.0)
            coverages.append(0.0)
            passes.append(0)
            retrieved_example_ids.append([["Timeout"] * k])
            retrieved_example_distances.append([["Timeout"] * k])
            prompt.append({"messages": []})

        n = len(generated_cyphers)
        if every and (n % every) == 0:
            df_ckpt = pd.DataFrame({
                "generated_cypher": generated_cyphers,
                "true_data": true_datas,
                "eval_data": eval_datas,
                "jaro_winkler": jaro_winklers,
                "jaccard": jaccards,
                "coverage": coverages,
                "pass": passes,
                "retrieved_example_ids": retrieved_example_ids,
                "retrieved_example_distances": retrieved_example_distances,
                "prompt": prompt,
            }, index=df.index[:n])
            save_checkpoint(df_ckpt, "eval_checkpoint.csv")

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverage
    out["pass"] = passes
    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances
    out["prompt"] = prompt
    return out


inputs = {
    "question": "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"
}

enriched = enrich_with_examples_with_output(inputs)

prompt_messages = cypher_prompt.format_messages(
    question=enriched["question"],
    examples=enriched["examples"]
)

print("=== PROMPT ===")
for m in prompt_messages:
    print(f"\n[{m.type.upper()}]\n{m.content}")
df = evaluate_cypher_queries_RAG_with_output(test_df, graph, cypher_generate, cypher_prompt)
df.head(3)

df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_RAG+output.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_RAG+output.pkl")
df.head(n=3)

***
# Schema + Examples

Let's combine enhanced schema and examples. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
# Generate Cypher statement based on natural language input

cypher_template = """Based on the Neo4j graph schema below,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.
{enhanced_schema}

Examples:
The following examples provide useful patterns for querying the graph database.
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_template(cypher_template)

cypher_chain = (
    RunnablePassthrough.assign(
        enhanced_schema=lambda _: enhanced_schema,
        examples=lambda x: retrieve_examples_bundle(x["question"])
    )
    | cypher_prompt
    | llm
    | StrOutputParser()
)

def evaluate_cypher_queries_RAG_schema(df: pd.DataFrame, graph, cypher_generate, cypher_prompt, schema, k: int = 1, every=10) -> pd.DataFrame:
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    
    prompt = []
    i = 0

    retrieved_example_ids = []
    retrieved_example_distances = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            with time_limit(200):
                true_data = _run_graph_query_safe(graph, _normalize_generated_cypher(row["cypher"]))

                example_generated_cyphers = []
                example_eval_datas = []
                example_ids_runs = []
                example_dist_runs = []
                prompts = []

                for _ in range(k):
                    enriched = enrich_with_examples({"question": row["question"]})

                    
                    example_ids_runs.append(enriched["example_ids"])
                    example_dist_runs.append(enriched["example_distances"])

                    with torch.inference_mode():

                        formatted_prompt = cypher_prompt.format_messages(
                            question=enriched["question"],
                            examples=enriched["examples"],
                            enhanced_schema=schema
                        )
                        prompts.append({
                            "messages": [
                                {"role": msg.type, "content": msg.content}
                                for msg in formatted_prompt
                            ],
                        })

                        cypher = _normalize_generated_cypher(cypher_chain.invoke(
                            {"question": enriched["question"],
                            "examples": enriched["examples"]}
                        ).rsplit("\nCypher query: ", 1)[-1])

                    example_generated_cyphers.append(cypher)
                    example_eval_datas.append(_run_graph_query_safe(graph, cypher))
                        
                gc.collect()
                torch.cuda.empty_cache()
                
                jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])

                jaccard = jaccard_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                )

                coverage = coverage_df_sim_pair(
                    (row["cypher"], true_data),
                    (example_generated_cyphers[0], example_eval_datas[0])
                )
                passs = 1 if coverage == 1 else 0

                # append
                generated_cyphers.append(example_generated_cyphers)
                true_datas.append(true_data)
                eval_datas.append(example_eval_datas)
                jaro_winklers.append(jw_value)
                jaccards.append(jaccard)
                coverages.append(coverage)
                passes.append(passs)
                
                prompt.append(prompts)

                retrieved_example_ids.append(example_ids_runs)
                retrieved_example_distances.append(example_dist_runs)

                if (i % every) == 0:
                    n = len(generated_cyphers)
                    df_ckpt = pd.DataFrame({
                        "generated_cypher": generated_cyphers,
                        "true_data": true_datas,
                        "eval_data": eval_datas,
                        "jaro_winkler": jaro_winklers,
                        "jaccard": jaccards,
                        "coverage": coverages,
                        "pass": passes,
                        
                        "retrieved_example_ids": retrieved_example_ids,
                        "retrieved_example_distances": retrieved_example_distances,
                        "prompt": prompt,
                    }, index=df.index[:n])  
                    save_checkpoint(df_ckpt, "eval_checkpoint.csv")
                i += 1
        except TimeoutError as e:
            generated_cyphers.append(["Timeout"] * k)
            true_datas.append([{"id": "Timeout"}])
            eval_datas.append([{"id": "Timeout"}] * k)
            jaro_winklers.append(0.0)
            jaccards.append(0.0)
            coverages.append(0.0)
            passes.append(0)
            
            retrieved_example_ids.append([["Timeout"] * k])
            retrieved_example_distances.append([["Timeout"] * k])
            prompt.append({"messages": []})
            i += 1

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverage
    out["pass"] = passes
    
    out["prompt"] = prompt

    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances

    return out

df = evaluate_cypher_queries_RAG_schema(test_df, graph, cypher_chain, cypher_prompt, enhanced_schema)
df.head(n=3)

df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_schema+RAG.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_schema+RAG.pkl")
df.head(n=3)

***
# Schema + Examples + Output

Let's combine enhanced schema, examples, and their outputs. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
# Generate Cypher statement based on natural language input

cypher_template = """Based on the Neo4j graph schema below,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.
{enhanced_schema}

Examples:
The following examples provide useful patterns for querying
the graph database (Neo4j outputs are also reported).
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_template(cypher_template)

cypher_chain = (
    RunnablePassthrough.assign(
        enhanced_schema=lambda _: enhanced_schema,
        examples=lambda x: retrieve_examples_bundle_with_output(x["question"])
    )
    | cypher_prompt
    | llm
    | StrOutputParser()
)

def evaluate_cypher_queries_RAG_with_output_schema(df, graph, cypher_generate, cypher_prompt, schema, k=1, every=10):
    generated_cyphers, true_datas, eval_datas = [], [], []
    jaro_winklers, jaccards, coverages = [], [], []
    passes = []
    retrieved_example_ids, retrieved_example_distances, prompt = [], [], []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            with time_limit(200):
                true_data = _run_graph_query_safe(graph, row["cypher"])

                example_generated_cyphers, example_eval_datas = [], []
                example_ids_runs, example_dist_runs, prompts = [], [], []

                for _ in range(k):
                    enriched = enrich_with_examples_with_output({"question": row["question"]})
                    example_ids_runs.append(enriched["example_ids"])
                    example_dist_runs.append(enriched["example_distances"])

                    with torch.inference_mode():

                        formatted_prompt = cypher_prompt.format_messages(
                            question=enriched["question"],
                            examples=enriched["examples"],
                            enhanced_schema=schema
                        )
                        prompts.append({
                            "messages": [
                                {"role": msg.type, "content": msg.content}
                                for msg in formatted_prompt
                            ],
                        })

                        cypher = cypher_chain.invoke(
                            {"question": enriched["question"], "examples": enriched["examples"]}
                        ).rsplit("\nCypher query: ", 1)[-1]

                    example_generated_cyphers.append(cypher)
                    example_eval_datas.append(_run_graph_query_safe(graph, cypher))

                jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])
                jaccard = jaccard_df_sim_pair((row["cypher"], true_data),
                                      (example_generated_cyphers[0], example_eval_datas[0]))
                coverage = coverage_df_sim_pair((row["cypher"], true_data),
                                              (example_generated_cyphers[0], example_eval_datas[0]))

                passs = 1 if coverage == 1 else 0

                generated_cyphers.append(example_generated_cyphers)
                true_datas.append(true_data)
                eval_datas.append(example_eval_datas)
                jaro_winklers.append(jw_value)
                jaccards.append(jaccard)
                coverages.append(coverage)
                passes.append(passs)
                retrieved_example_ids.append(example_ids_runs)
                retrieved_example_distances.append(example_dist_runs)
                prompt.append(prompts)

        except TimeoutError:
            generated_cyphers.append(["Timeout"] * k)
            true_datas.append([{"id": "Timeout"}])
            eval_datas.append([{"id": "Timeout"}] * k)
            jaro_winklers.append(0.0)
            jaccards.append(0.0)
            coverages.append(0.0)
            passes.append(0)
            retrieved_example_ids.append([["Timeout"] * k])
            retrieved_example_distances.append([["Timeout"] * k])
            prompt.append({"messages": []})

        n = len(generated_cyphers)
        if every and (n % every) == 0:
            df_ckpt = pd.DataFrame({
                "generated_cypher": generated_cyphers,
                "true_data": true_datas,
                "eval_data": eval_datas,
                "jaro_winkler": jaro_winklers,
                "jaccard": jaccards,
                "coverage": coverages,
                "pass": passes,
                "retrieved_example_ids": retrieved_example_ids,
                "retrieved_example_distances": retrieved_example_distances,
                "prompt": prompt,
            }, index=df.index[:n])
            save_checkpoint(df_ckpt, "eval_checkpoint.csv")

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverage
    out["pass"] = passes
    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances
    out["prompt"] = prompt
    return out



df = evaluate_cypher_queries_RAG_with_output_schema(test_df, graph, cypher_chain, cypher_prompt, enhanced_schema)
df.head(n=3)

df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_schema+RAG+output.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_schema+RAG+output.pkl")
df.head(n=3)